Cross-attention

In [29]:
import numpy as np
np.random.seed(42)

def tokenize(text):
    print(f"TOKENIZA ORACIONES: {text}")
    return text.lower().replace(".", "").split()

def build_vocab(sentences):
    print("ARMAR VOCABULARIO")
    vocab = {}
    idx = 0
    for sent in sentences:
        for word in tokenize(sent):
            if word not in vocab:
                vocab[word] = idx
                idx += 1
    return vocab

def create_embeddings(vocab_size, d_model):
    print(f"CREAR EMBEDDINGS {vocab_size}, {np.random.randn(vocab_size, d_model).shape}")
    return np.random.randn(vocab_size, d_model)

def softmax(x, axis=-1):
    x = x - np.max(x, axis=axis, keepdims=True)
    exp_x = np.exp(x)
    weights = exp_x / np.sum(exp_x, axis=axis, keepdims=True)
    print(f"Attention weights: {weights.shape}")
    return weights

def self_attention(X, Wq, Wk, Wv):
    print("APLICAR SELF ATTENTION")
    Q = X @ Wq
    K = X @ Wk
    V = X @ Wv
    print(f"Q: {Q.shape}, K: {K.shape}, V: {V.shape}")
    d_k = Q.shape[-1]
    scores = (Q @ K.T) / np.sqrt(d_k)
    print(f"Q @ K.T {(Q @ K.T).shape}, scores: {scores.shape}")
    weights = softmax(scores, axis=-1)
    
    output = weights @ V
    print(f"weights @ V: {weights.shape} @ {V.shape} = {(weights @ V).shape}, Output: {output.shape}, Attn: {weights.shape}")
    return output, weights

def mean_pooling(H):
    mean = []
    for j in range(H.shape[1]):
        sum = 0
        for i in range(H.shape[0]):
            sum += H[i][j]
        mean.append(sum / H.shape[0])
    return np.array(mean)


def cross_encoder(sentence_a, sentence_b, vocab, embedding_matrix, Wq, Wk, Wv, W_out):
    tokens_a = tokenize(sentence_a)
    tokens_b = tokenize(sentence_b)
    tokens = tokens_a + ["[SEP]"] + tokens_b
    print(tokens_a, "SEP", tokens_b)
    print(len(tokens_a), 1, len(tokens_b), "=", len(tokens_a) + 1 + len(tokens_b))
    
    if "[SEP]" not in vocab:
        vocab = vocab.copy()
        vocab["[SEP]"] = len(vocab)
        embedding_matrix = np.vstack([ embedding_matrix, np.random.randn(embedding_matrix.shape[1])])    
    
    X = np.array([embedding_matrix[vocab[t]] for t in tokens])
    print("X.shape", X.shape)

    H, attn = self_attention(X, Wq, Wk, Wv)
    # print(H)
    print(f"H: {H.shape}, attn: {attn.shape}")

    pooled = mean_pooling(H)
    print(f"POOLED: {pooled}, shape: {pooled.shape}")
    print(f"W_out: {W_out}, shape: {W_out.shape}")
    # print(f"NUMPY: {H.mean(axis=0)}")
    score = pooled @ W_out
    print(f"pooled @ W_out: {pooled.shape} @ {W_out.shape} = {(pooled @ W_out).shape}, Score: {score}")
    return score, attn, tokens

candidate = "The cat sleeps peacefully under the afternoon sun."
targets = [
    "The dog barks when someone knocks on the door.",
    "The afternoon is warm and perfect for resting.",
    "Tomorrow I have to submit the final project.",
    "Cats usually look for sunny places to sleep.",
    "Programming in JavaScript can be fun."
]

all_sentences = [candidate] + targets
vocab = build_vocab(all_sentences)

d_model = 8
d_k = 8

embedding_matrix = create_embeddings(len(vocab), d_model)
print(f"Embedding matrix shape: {embedding_matrix.shape}")

Wq = np.random.randn(d_model, d_k)
Wk = np.random.randn(d_model, d_k)
Wv = np.random.randn(d_model, d_k)
W_out = np.random.randn(d_k) # qué dimensiones del vector importan para decidir relevancia

results = []

for target in targets:
    print("==================")
    score, attn, tokens = cross_encoder(
        candidate, target,
        vocab, embedding_matrix,
        Wq, Wk, Wv, W_out
    )
    print("==================")
    results.append((target, score))

results_sorted = sorted(results, key=lambda x: x[1], reverse=True)

for i, (target, score) in enumerate(results_sorted, 1):
    print(f"{i}. Score: {score:.4f}")
    print(f"   {target}")

ARMAR VOCABULARIO
TOKENIZA ORACIONES: The cat sleeps peacefully under the afternoon sun.
TOKENIZA ORACIONES: The dog barks when someone knocks on the door.
TOKENIZA ORACIONES: The afternoon is warm and perfect for resting.
TOKENIZA ORACIONES: Tomorrow I have to submit the final project.
TOKENIZA ORACIONES: Cats usually look for sunny places to sleep.
TOKENIZA ORACIONES: Programming in JavaScript can be fun.
CREAR EMBEDDINGS 39, (39, 8)
Embedding matrix shape: (39, 8)
TOKENIZA ORACIONES: The cat sleeps peacefully under the afternoon sun.
TOKENIZA ORACIONES: The dog barks when someone knocks on the door.
['the', 'cat', 'sleeps', 'peacefully', 'under', 'the', 'afternoon', 'sun'] SEP ['the', 'dog', 'barks', 'when', 'someone', 'knocks', 'on', 'the', 'door']
8 1 9 = 18
X.shape (18, 8)
APLICAR SELF ATTENTION
Q: (18, 8), K: (18, 8), V: (18, 8)
Q @ K.T (18, 18), scores: (18, 18)
Attention weights: (18, 18)
weights @ V: (18, 18) @ (18, 8) = (18, 8), Output: (18, 8), Attn: (18, 18)
H: (18, 8), at

In [34]:
import numpy as np
np.random.seed(42)

def tokenize(text):
    print(f"TOKENIZA ORACION: {text}")
    return text.lower().replace(".", "").split()

def build_vocab(sentences, special_tokens=None):
    print("ARMAR VOCABULARIO")
    vocab = {}
    idx = 0

    if special_tokens is not None:
        print(f"AGREGAR TOKENS ESPECIALES: {special_tokens}")
        for tok in special_tokens:
            if tok not in vocab:
                vocab[tok] = idx
                idx += 1

    for sent in sentences:
        for word in tokenize(sent):
            if word not in vocab:
                vocab[word] = idx
                idx += 1

    print(f"VOCAB SIZE: {len(vocab)}")
    return vocab

def create_embeddings(vocab_size, d_model):
    embedding_matrix = np.random.randn(vocab_size, d_model)
    print(f"CREAR EMBEDDINGS: vocab_size={vocab_size}, d_model={d_model}")
    print(f"EMBEDDING MATRIX SHAPE: {embedding_matrix.shape}")
    return embedding_matrix

def softmax(x, axis=-1):
    print(f"SOFTMAX INPUT SHAPE: {x.shape}")
    x = x - np.max(x, axis=axis, keepdims=True)
    exp_x = np.exp(x)
    weights = exp_x / np.sum(exp_x, axis=axis, keepdims=True)
    print(f"SOFTMAX OUTPUT SHAPE: {weights.shape}")
    return weights

def self_attention(X, Wq, Wk, Wv):
    print("APLICAR SELF-ATTENTION")
    print(f"X SHAPE: {X.shape}")
    print(f"Wq SHAPE: {Wq.shape}")
    print(f"Wk SHAPE: {Wk.shape}")
    print(f"Wv SHAPE: {Wv.shape}")

    Q = X @ Wq
    K = X @ Wk
    V = X @ Wv

    print(f"Q = X @ Wq -> {X.shape} @ {Wq.shape} = {Q.shape}")
    print(f"K = X @ Wk -> {X.shape} @ {Wk.shape} = {K.shape}")
    print(f"V = X @ Wv -> {X.shape} @ {Wv.shape} = {V.shape}")

    d_k = Q.shape[-1]
    print(f"d_k: {d_k}")

    raw_scores = Q @ K.T
    print(f"Q @ K.T -> {Q.shape} @ {K.T.shape} = {raw_scores.shape}")

    scores = raw_scores / np.sqrt(d_k)
    print(f"scores SHAPE: {scores.shape}")

    weights = softmax(scores, axis=-1)
    print(f"ATTENTION WEIGHTS SHAPE: {weights.shape}")

    output = weights @ V
    print(f"OUTPUT = weights @ V -> {weights.shape} @ {V.shape} = {output.shape}")

    return output, weights

def cross_encoder(sentence_a, sentence_b, vocab, embedding_matrix, Wq, Wk, Wv, W_out, b_out):
    print("\n" + "=" * 100)
    print("CROSS-ENCODER")

    tokens_a = tokenize(sentence_a)
    tokens_b = tokenize(sentence_b)

    tokens = ["[CLS]"] + tokens_a + ["[SEP]"] + tokens_b + ["[SEP]"]

    print(f"TOKENS A ({len(tokens_a)}): {tokens_a}")
    print(f"TOKENS B ({len(tokens_b)}): {tokens_b}")
    print(f"TOKENS FINALES ({len(tokens)}): {tokens}")
    print(f"VERIFICACION LONGITUD: 1 + {len(tokens_a)} + 1 + {len(tokens_b)} + 1 = {len(tokens)}")

    indices = [vocab[t] for t in tokens]
    print(f"INDICES EN VOCAB: {indices}")
    print(f"NUM TOKENS = {len(indices)}")

    X = np.array([embedding_matrix[vocab[t]] for t in tokens])
    print(f"X SHAPE: {X.shape}")
    print(f"ESPERADO: ({len(tokens)}, {embedding_matrix.shape[1]})")

    H, attn = self_attention(X, Wq, Wk, Wv)
    print(f"H SHAPE: {H.shape}")
    print(f"ATTN SHAPE: {attn.shape}")

    cls_vector = H[0]
    print(f"CLS VECTOR = H[0]")
    print(f"CLS VECTOR SHAPE: {cls_vector.shape}")
    print(f"ESPERADO: ({H.shape[1]},)")

    print(f"W_out SHAPE: {W_out.shape}")
    print(f"b_out TYPE: {type(b_out)}, VALUE: {b_out}")

    score = cls_vector @ W_out + b_out
    print(f"SCORE = cls_vector @ W_out + b_out")
    print(f"{cls_vector.shape} @ {W_out.shape} + scalar -> score")
    print(f"SCORE TYPE: {type(score)}")
    print(f"SCORE: {score}")

    return score, attn, tokens, H

candidate = "The cat sleeps peacefully under the afternoon sun."
targets = [
    "The dog barks when someone knocks on the door.",
    "The afternoon is warm and perfect for resting.",
    "Tomorrow I have to submit the final project.",
    "Cats usually look for sunny places to sleep.",
    "Programming in JavaScript can be fun."
]

special_tokens = ["[CLS]", "[SEP]"]
all_sentences = [candidate] + targets

vocab = build_vocab(all_sentences, special_tokens=special_tokens)

d_model = 8
d_k = 8

embedding_matrix = create_embeddings(len(vocab), d_model)

Wq = np.random.randn(d_model, d_k)
Wk = np.random.randn(d_model, d_k)
Wv = np.random.randn(d_model, d_k)

print("\nCREAR PESOS DE ATTENTION")
print(f"Wq SHAPE: {Wq.shape}")
print(f"Wk SHAPE: {Wk.shape}")
print(f"Wv SHAPE: {Wv.shape}")

W_out = np.random.randn(d_k)
b_out = np.random.randn()

print("\nCREAR CLASIFICADOR")
print(f"W_out SHAPE: {W_out.shape}")
print(f"b_out TYPE: {type(b_out)}, VALUE: {b_out}")

results = []

for target in targets:
    score, attn, tokens, H = cross_encoder(
        candidate, target,
        vocab, embedding_matrix,
        Wq, Wk, Wv,
        W_out, b_out
    )
    results.append((target, score))

print("\n" + "=" * 100)
print("RANKING FINAL")

results_sorted = sorted(results, key=lambda x: x[1], reverse=True)

for i, (target, score) in enumerate(results_sorted, 1):
    print(f"{i}. SCORE: {score:.4f}")
    print(f"   {target}")

ARMAR VOCABULARIO
AGREGAR TOKENS ESPECIALES: ['[CLS]', '[SEP]']
TOKENIZA ORACION: The cat sleeps peacefully under the afternoon sun.
TOKENIZA ORACION: The dog barks when someone knocks on the door.
TOKENIZA ORACION: The afternoon is warm and perfect for resting.
TOKENIZA ORACION: Tomorrow I have to submit the final project.
TOKENIZA ORACION: Cats usually look for sunny places to sleep.
TOKENIZA ORACION: Programming in JavaScript can be fun.
VOCAB SIZE: 41
CREAR EMBEDDINGS: vocab_size=41, d_model=8
EMBEDDING MATRIX SHAPE: (41, 8)

CREAR PESOS DE ATTENTION
Wq SHAPE: (8, 8)
Wk SHAPE: (8, 8)
Wv SHAPE: (8, 8)

CREAR CLASIFICADOR
W_out SHAPE: (8,)
b_out TYPE: <class 'float'>, VALUE: 0.25972250172148187

CROSS-ENCODER
TOKENIZA ORACION: The cat sleeps peacefully under the afternoon sun.
TOKENIZA ORACION: The dog barks when someone knocks on the door.
TOKENS A (8): ['the', 'cat', 'sleeps', 'peacefully', 'under', 'the', 'afternoon', 'sun']
TOKENS B (9): ['the', 'dog', 'barks', 'when', 'someone',